# 03 — Results Aggregation (PGR207 CIFAR-10 mid-term)

**Student IDs:** `<STUDENT_ID_1>`, `<STUDENT_ID_2>` *(fill in before submitting)*

Loads `all_results.csv` (21 rows, one per training run, produced by
`02_full_experiment.ipynb`) and produces the tables/figures the paper needs:

1. One summary table per factor study (architecture, augmentation, optimizer), each showing
   the baseline `C1` plus its two study-specific variants, as **mean +/- std over the 3 seeds**.
2. Confusion matrices (element-wise mean over the 3 seeds) and per-class precision/recall for
   the best and worst configurations by mean test accuracy.
3. Training curves (train/val loss and accuracy per epoch) for the runs discussed in the paper.

All metrics here are computed **per run, then averaged** — never by pooling predictions across
seeds — per the brief's explicit requirement.


## 1. Load results

In [ ]:
import pipeline_config as cfg
import pipeline_metrics as metrics

raw_df = metrics.load_results(cfg.RESULTS_CSV_DEFAULT)
assert len(raw_df) == 7 * 3, f"Expected 21 rows, found {len(raw_df)} -- did all runs in 02 finish?"

df = metrics.decode_json_columns(raw_df)   # parses the JSON-string columns back into dicts/lists
df[["config_id", "architecture", "augmentation", "optimizer", "seed", "test_accuracy", "test_macro_f1"]]


## 2. Per-configuration mean +/- std over seeds

The single source table everything below is built from.


In [ ]:
summary = metrics.aggregate_over_seeds(raw_df)
summary


## 3. The three factor-study tables

Each table has exactly 3 rows (baseline + 2 variants) and matches the shape required by
Section 3.5 / the Results section of the paper. Report the **difference next to the
seed-to-seed std** — if the gap between two rows is within about one std of either row, say so
explicitly rather than declaring a winner.


In [ ]:
architecture_table = metrics.build_factor_table(summary, "architecture")
architecture_table


In [ ]:
augmentation_table = metrics.build_factor_table(summary, "augmentation")
augmentation_table


In [ ]:
optimizer_table = metrics.build_factor_table(summary, "optimizer")
optimizer_table


### Formatted mean +/- std strings (for pasting into the paper's tables)


In [ ]:
def format_table(table):
    out = table.copy()
    out["accuracy_%"] = (
        (out["accuracy_mean"] * 100).round(1).astype(str)
        + " +/- " + (out["accuracy_std"] * 100).round(1).astype(str)
    )
    out["macro_f1"] = (
        out["macro_f1_mean"].round(3).astype(str)
        + " +/- " + out["macro_f1_std"].round(3).astype(str)
    )
    return out[["config_id", "architecture", "augmentation", "optimizer", "num_params", "accuracy_%", "macro_f1"]]

print("Architecture study:")
display(format_table(architecture_table))
print("\nAugmentation study:")
display(format_table(augmentation_table))
print("\nOptimizer study:")
display(format_table(optimizer_table))


## 4. Best and worst configurations (by mean test accuracy)


In [ ]:
best_id, worst_id = metrics.best_worst_config_ids(summary)
print(f"Best config:  {best_id}  (acc = {summary.set_index('config_id').loc[best_id, 'accuracy_mean']:.3f})")
print(f"Worst config: {worst_id}  (acc = {summary.set_index('config_id').loc[worst_id, 'accuracy_mean']:.3f})")


## 5. Confusion matrices for best/worst configs (mean over 3 seeds)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_mean_confusion_matrix(df, config_id, ax):
    cms = df.loc[df["config_id"] == config_id, "confusion_matrix"].tolist()
    mean_cm = metrics.mean_confusion_matrix(cms)
    # Normalize by true-class row totals so the plot shows recall-per-cell, easier to read
    # across classes with equal support (CIFAR-10 is balanced, so this is just a display choice).
    norm_cm = mean_cm / mean_cm.sum(axis=1, keepdims=True)

    im = ax.imshow(norm_cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(cfg.CLASS_NAMES)))
    ax.set_yticks(range(len(cfg.CLASS_NAMES)))
    ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(cfg.CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{config_id} (mean over {len(cms)} seeds)")
    return im, mean_cm


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
_, best_cm = plot_mean_confusion_matrix(df, best_id, axes[0])
im, worst_cm = plot_mean_confusion_matrix(df, worst_id, axes[1])
fig.colorbar(im, ax=axes, fraction=0.03, label="Row-normalized frequency")
fig.suptitle("Confusion matrices: best vs. worst configuration")
plt.savefig("confusion_best_worst.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Per-class precision/recall for best/worst configs (mean over 3 seeds)


In [ ]:
import pandas as pd


def per_class_table(df, config_id):
    rows = df.loc[df["config_id"] == config_id]
    precision = pd.DataFrame(rows["precision_per_class"].tolist())
    recall = pd.DataFrame(rows["recall_per_class"].tolist())
    f1 = pd.DataFrame(rows["f1_per_class"].tolist())
    out = pd.DataFrame({
        "precision_mean": precision.mean(), "precision_std": precision.std(),
        "recall_mean": recall.mean(), "recall_std": recall.std(),
        "f1_mean": f1.mean(), "f1_std": f1.std(),
    })
    out.index.name = "class"
    return out.round(3)


print(f"Per-class metrics -- {best_id} (best):")
display(per_class_table(df, best_id))
print(f"\nPer-class metrics -- {worst_id} (worst):")
display(per_class_table(df, worst_id))


## 7. Training curves

Per-epoch train/val loss and accuracy for a chosen run — the main evidence for statements
about convergence speed and overfitting. Change `curve_config_id` / `curve_seed` to inspect any
of the 21 runs.


In [ ]:
curve_config_id = "C1"  # the baseline; change to inspect any other config
curve_seed = 0

row = df[(df["config_id"] == curve_config_id) & (df["seed"] == curve_seed)].iloc[0]
history = row["history"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
axes[0].set_title(f"{curve_config_id} seed={curve_seed} -- loss")

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()
axes[1].set_title(f"{curve_config_id} seed={curve_seed} -- accuracy")

plt.tight_layout()
plt.savefig(f"training_curves_{curve_config_id}_seed{curve_seed}.png", dpi=150)
plt.show()


## 8. Training cost vs. accuracy (parameter count and wall-clock time)

Useful for the "accuracy per parameter / per unit of training time" question in Section 3.7
of the brief.


In [ ]:
cost_table = summary[["config_id", "architecture", "num_params", "train_time_mean", "accuracy_mean", "accuracy_std"]]
cost_table = cost_table.sort_values("accuracy_mean", ascending=False)
cost_table


## Notes for the write-up

- Averaging convention: **macro** averaging for F1/precision/recall throughout (every class
  weighted equally) — stated once here, kept consistent everywhere, per `compute_metrics` in
  `pipeline_metrics.py`.
- Confusion matrices above are the **element-wise mean over the 3 seeds** (not a single
  representative seed) — say this explicitly in the paper's caption.
- Any two configurations should only be compared if they differ in exactly one factor (check
  `cfg.CONFIGS` / `cfg.FACTOR_STUDIES`) — `C1` is intentionally repeated across all three factor
  tables above, since it is the shared baseline, not a duplication error.
- When a gap between two rows in a factor table is smaller than roughly one seed-to-seed std,
  report it as a tie within noise rather than a winner.